# Feature engineering for document retrieval

This notebook covers text preprocessing, TF-IDF vectorization, BM25 index construction,
and analysis of the resulting feature representations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

from src.data_loader import load_documents, build_chunk_index, preprocess_text
from src.model import TfidfRetriever, BM25Retriever

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
GOLD = '#E8C230'
NAVY = '#3B6FD4'

documents = load_documents('../data/documents.json')
chunks, metadata = build_chunk_index(documents, chunk_size=500, overlap=50)
print(f'Documents: {len(documents)}')
print(f'Chunks: {len(chunks)}')

## 1. Text preprocessing

Before vectorization, we apply basic text cleaning: lowercasing, removing special characters, and collapsing whitespace. We compare raw and preprocessed text to verify the cleaning does not remove important content.

In [ ]:
# Show preprocessing effect on a sample chunk
sample = chunks[0]
cleaned = preprocess_text(sample)

print('=== Raw chunk ===')
print(sample[:300])
print(f'\n=== Preprocessed ===')
print(cleaned[:300])
print(f'\nRaw length:    {len(sample)} chars, {len(sample.split())} words')
print(f'Cleaned length: {len(cleaned)} chars, {len(cleaned.split())} words')

## 2. TF-IDF vectorization

We build a TF-IDF matrix using scikit-learn's TfidfVectorizer. Key parameters:
- **max_features=5000**: limit vocabulary to the 5000 most important terms
- **ngram_range=(1,2)**: include both unigrams and bigrams to capture phrases like "property tax" or "transit ridership"
- **sublinear_tf=True**: apply logarithmic term frequency scaling to prevent very frequent terms from dominating
- **stop_words='english'**: remove common English stop words

In [ ]:
tfidf = TfidfRetriever(max_features=5000, ngram_range=(1, 2))
tfidf.fit(chunks, metadata)

feature_names = tfidf.get_feature_names()
tfidf_matrix = tfidf.doc_vectors

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  {tfidf_matrix.shape[0]} chunks x {tfidf_matrix.shape[1]} features')
print(f'Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%}')
print(f'Non-zero entries: {tfidf_matrix.nnz:,}')

In [ ]:
# Top TF-IDF terms per chunk (first 5 chunks)
for i in range(min(5, len(chunks))):
    row = tfidf_matrix[i].toarray().flatten()
    top_indices = np.argsort(row)[::-1][:8]
    top_terms = [(feature_names[j], row[j]) for j in top_indices if row[j] > 0]
    doc_id = metadata[i]['doc_id']
    print(f'Chunk {i} ({doc_id}): {", ".join(f"{t}={s:.3f}" for t, s in top_terms)}')

## 3. TF-IDF weight distribution

The distribution of TF-IDF weights tells us how informative the features are. A healthy distribution should have most weights near zero (most terms are irrelevant to most chunks) with a long tail of high-weight terms that drive retrieval.

In [ ]:
nonzero_weights = tfidf_matrix.data

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(nonzero_weights, bins=50, color=NAVY, alpha=0.7, edgecolor='white')
axes[0].set_xlabel('TF-IDF weight')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of non-zero TF-IDF weights')

# IDF values (inverse document frequency)
idf_values = tfidf.vectorizer.idf_
axes[1].hist(idf_values, bins=30, color=GOLD, alpha=0.7, edgecolor='white')
axes[1].set_xlabel('IDF value')
axes[1].set_ylabel('Number of terms')
axes[1].set_title('IDF distribution across vocabulary')

plt.tight_layout()
plt.savefig('../figures/tfidf_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'IDF range: {idf_values.min():.2f} to {idf_values.max():.2f}')
print(f'Mean IDF: {idf_values.mean():.2f}')

## 4. Chunk similarity matrix

The pairwise cosine similarity between all chunks reveals how similar different passages are to each other. Chunks from the same document should be more similar to each other than to chunks from unrelated documents. High inter-document similarity between specific pairs might indicate overlapping policy areas that could confuse retrieval.

In [ ]:
sim_matrix = cosine_similarity(tfidf_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='YlOrBr', vmin=0, vmax=1)
ax.set_xlabel('Chunk index')
ax.set_ylabel('Chunk index')
ax.set_title('Pairwise cosine similarity between chunks (TF-IDF)')
plt.colorbar(im, ax=ax, label='Cosine similarity')

# Add document boundary lines
chunk_boundaries = []
current_doc = metadata[0]['doc_id']
for i, m in enumerate(metadata):
    if m['doc_id'] != current_doc:
        chunk_boundaries.append(i)
        current_doc = m['doc_id']

for b in chunk_boundaries:
    ax.axhline(b - 0.5, color='white', linewidth=0.5, alpha=0.7)
    ax.axvline(b - 0.5, color='white', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.savefig('../figures/chunk_similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. BM25 index construction

BM25 (Best Matching 25) is an alternative to TF-IDF that uses term frequency saturation and document length normalization. The key parameters are:
- **k1**: controls term frequency saturation (higher = slower saturation)
- **b**: controls document length normalization (0 = no normalization, 1 = full normalization)

In [ ]:
bm25 = BM25Retriever(k1=1.5, b=0.75)
bm25.fit(chunks, metadata)

# Compare TF-IDF and BM25 scores for a sample query
sample_query = "What is the affordable housing target?"
tfidf_results = tfidf.retrieve(sample_query, k=5)
bm25_results = bm25.retrieve(sample_query, k=5)

print(f'Query: "{sample_query}"\n')
print('TF-IDF top 5:')
for r in tfidf_results:
    print(f'  {r["metadata"]["doc_id"]} chunk {r["metadata"]["chunk_idx"]}: '
          f'score={r["score"]:.4f} | {r["chunk"][:80]}...')

print(f'\nBM25 top 5:')
for r in bm25_results:
    print(f'  {r["metadata"]["doc_id"]} chunk {r["metadata"]["chunk_idx"]}: '
          f'score={r["score"]:.4f} | {r["chunk"][:80]}...')

## 6. Feature importance: highest IDF terms

Terms with the highest IDF values are the most discriminative -- they appear in the fewest documents and carry the most information when they do appear. These are the terms that drive retrieval for specific queries.

In [ ]:
idf_values = tfidf.vectorizer.idf_
sorted_idf = np.argsort(idf_values)[::-1]

print('Top 20 highest-IDF terms (most discriminative):')
for i in sorted_idf[:20]:
    print(f'  {feature_names[i]:30s} IDF={idf_values[i]:.3f}')

print(f'\nTop 20 lowest-IDF terms (least discriminative):')
for i in sorted_idf[-20:]:
    print(f'  {feature_names[i]:30s} IDF={idf_values[i]:.3f}')

## 7. Bigram analysis

Bigrams (two-word phrases) can capture meaning that individual words miss. For example, "property tax" is more specific than either "property" or "tax" alone. We examine which bigrams the TF-IDF model assigns the highest weights.

In [ ]:
bigrams = [(f, idf_values[i]) for i, f in enumerate(feature_names) if ' ' in f]
bigrams.sort(key=lambda x: x[1], reverse=True)

print(f'Total bigram features: {len(bigrams)}')
print(f'\nTop 25 bigrams by IDF:')
for term, idf in bigrams[:25]:
    print(f'  {term:35s} IDF={idf:.3f}')

# Plot top 20 bigrams
top_bg = bigrams[:20]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([t for t, _ in top_bg], [v for _, v in top_bg], color=GOLD, alpha=0.85)
ax.set_xlabel('IDF value')
ax.set_title('Top 20 bigrams by IDF (most discriminative phrases)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/top_bigrams.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Chunk overlap analysis

Because we use overlapping chunks, adjacent chunks from the same document share some text. We measure how similar consecutive chunks are to confirm that the overlap is working as intended: providing continuity without creating near-duplicates.

In [ ]:
# Measure similarity between consecutive chunks from the same document
consecutive_sims = []
for i in range(len(chunks) - 1):
    if metadata[i]['doc_id'] == metadata[i+1]['doc_id']:
        sim = cosine_similarity(tfidf_matrix[i], tfidf_matrix[i+1])[0, 0]
        consecutive_sims.append(sim)

# Measure similarity between random chunk pairs from different documents
import random
random.seed(42)
random_sims = []
for _ in range(100):
    i, j = random.sample(range(len(chunks)), 2)
    if metadata[i]['doc_id'] != metadata[j]['doc_id']:
        sim = cosine_similarity(tfidf_matrix[i], tfidf_matrix[j])[0, 0]
        random_sims.append(sim)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(consecutive_sims, bins=15, alpha=0.7, color=GOLD, label=f'Consecutive (same doc), mean={np.mean(consecutive_sims):.3f}')
ax.hist(random_sims, bins=15, alpha=0.7, color=NAVY, label=f'Random (diff doc), mean={np.mean(random_sims):.3f}')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('Chunk similarity: consecutive vs random pairs')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Summary

Key findings from feature engineering:

- **TF-IDF matrix**: The vectorizer produces a sparse matrix with thousands of features, dominated by bigram terms that capture municipal policy phrases
- **IDF distribution**: High-IDF terms like specific policy names and technical terms are the most useful for retrieval, while generic terms like "city" and "program" have low IDF
- **Chunk similarity**: Consecutive chunks from the same document show moderate overlap (as expected from the 50-character overlap setting), while chunks from different documents are well separated
- **BM25 vs TF-IDF**: Both methods retrieve similar results for most queries, but their score distributions differ due to BM25's term frequency saturation

Next steps: The modeling notebook will compare retrieval accuracy between TF-IDF and BM25 on the evaluation Q&A set.